In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import logging

logging.basicConfig(level=logging.ERROR)

In [2]:
from datasets import load_dataset, DownloadConfig

download_config = DownloadConfig(
    local_files_only=True,
    cache_dir=".cache",   # optional
)

LANGS = ['af_za', 'am_et', 'ar_eg', 'ast_es', 'az_az', 'be_by', 'bg_bg', 'bn_in', 'ca_es', 'ceb_ph', 'ckb_iq', 'cmn_hans_cn', 'cs_cz', 'cy_gb', 'da_dk', 'de_de', 'el_gr', 'en_us', 'es_419', 'et_ee', 'fa_ir', 'ff_sn', 'fi_fi', 'fr_fr', 'ga_ie', 'gl_es', 'ha_ng', 'he_il', 'hi_in', 'hr_hr', 'hu_hu', 'hy_am', 'id_id', 'it_it', 'ja_jp', 'jv_id', 'ka_ge', 'kk_kz', 'km_kh', 'kn_in', 'ko_kr', 'ky_kg', 'lg_ug', 'lo_la', 'lt_lt', 'lv_lv', 'mi_nz', 'mk_mk', 'ml_in', 'mn_mn', 'mr_in', 'ms_my', 'mt_mt', 'my_mm', 'nb_no', 'ne_np', 'nl_nl', 'ny_mw', 'om_et', 'or_in', 'pa_in', 'pl_pl', 'ps_af', 'pt_br', 'ro_ro', 'ru_ru', 'sl_si', 'sn_zw', 'so_so', 'sv_se', 'sw_ke', 'ta_in', 'te_in', 'tg_tj', 'th_th', 'tr_tr', 'uk_ua', 'ur_pk', 'uz_uz', 'vi_vn', 'wo_sn', 'xh_za', 'yo_ng', 'yue_hant_hk', 'zu_za']

/localscratch/nnsfn01/anaconda3/envs/north_caucasus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

model_path = "models/mms-300m-ipa-doreco"
if os.path.exists(model_path):
    processor = Wav2Vec2Processor.from_pretrained(model_path)
    model = Wav2Vec2ForCTC.from_pretrained(model_path)
    tokenizer = processor.tokenizer
else:
    processor = None
    model = None
    tokenizer = None

/localscratch/nnsfn01/anaconda3/envs/north_caucasus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from typing import List, Optional
import ctc_segmentation
import numpy as np
from transformers import Wav2Vec2Tokenizer
from lingpy.sequence.sound_classes import ipa2tokens

    
class CTCSegmentation:

    
    
    def __init__(self, tokenizer: Wav2Vec2Tokenizer= None, sampling_rate: int= 16000):
        self.tokenizer = tokenizer
        if tokenizer:
            char_list = [tokenizer.convert_ids_to_tokens(i) for i in range(tokenizer.vocab_size)]
            self.config = ctc_segmentation.CtcSegmentationParameters(char_list=char_list)
            self.sampling_rate = sampling_rate
        else:
            char_list = ['<unk>']
            self.config = ctc_segmentation.CtcSegmentationParameters(char_list=char_list)
            self.sampling_rate = 16_000
        self.total_failures = 0
        

    def get_word_and_timestamps_batch(
        self,
        probs_batch: List[np.ndarray],
        audio_lens: List[int],
        frame_lens: List[int],
        ipa_transcripts: Optional[List[str]],
        transcripts: Optional[List[str]],
    ) -> List[dict]:

        alignments = []
        
        
        # --- Transcript handling ---
        if not ipa_transcripts:
            pred_ids = probs_batch.argmax(axis=-1)
            ipa_transcripts = self.tokenizer.batch_decode(pred_ids)

        if not transcripts:
            transcripts = ipa_transcripts
            
        for probs, audio_len, frame_len, ipa_transcript, transcript in zip(probs_batch, audio_lens, frame_lens, ipa_transcripts, transcripts):
            
            
            self.config.index_duration = ( audio_len / self.sampling_rate ) / frame_len
            # =====================
            # WORD ALIGNMENT
            # =====================
            words_ipa = ipa_transcript.split()
            words = transcript.split()

            assert len(words) == len(words_ipa)
            

            try:
                gt_mat, utt_idx = ctc_segmentation.prepare_text(self.config, words_ipa)
                probs = probs[:frame_len]
                timings, char_probs, _ = ctc_segmentation.ctc_segmentation(
                    self.config, probs, gt_mat
                )
    
                word_segments = ctc_segmentation.determine_utterance_segments(
                    self.config, utt_idx, char_probs, timings, words_ipa
                )


            except:
                # tokenize all words into phones
                self.total_failures += 1
                all_word_phones = []
            
                for w_ipa in words_ipa:
                    if w_ipa != "<unk>":
                        phs = ipa2tokens(w_ipa, merge_vowels=False)
                    else:
                        phs = ["<unk>"]
            
                    all_word_phones.append(phs)
            
                phone_counts = [len(phs) for phs in all_word_phones]
                total_phones = sum(phone_counts)
            
                total_duration = frame_len * self.config.index_duration
            
                current = 0.0
                word_segments = []
            
                for n_ph in phone_counts:
                    dur = total_duration * (n_ph / total_phones)
            
                    word_segments.append(
                        (current, current + dur, 0.0)
                    )
            
                    current += dur


            word_out = [
                {
                    "text": w,
                    "start": float(p[0]),
                    "end": float(p[1]),
                    "conf": float(p[2]),
                }
                for w, p in zip(words, word_segments)
            ]

            # =====================
            # PHONE ALIGNMENT (hierarchical)
            # =====================
            
            phone_out = []
            
            PAD_FRAMES = 0
            
            for word, word_ipa, seg in zip(words, words_ipa, word_out):
            
                # ---------------------------------
                # convert word timestamps -> frames
                # ---------------------------------
                start_frame = int(seg["start"] / self.config.index_duration)
            
                end_frame = int(seg["end"] / self.config.index_duration)
            
                # add small context padding
                start_frame = max(0, start_frame - PAD_FRAMES)
                if probs is not None:
                    end_frame = min(len(probs), end_frame + PAD_FRAMES)
                else:
                    end_frame = end_frame
            
            
                
            
                # ---------------------------------
                # tokenize IPA word
                # ---------------------------------
                if word_ipa != '<unk>':
                    phones = ipa2tokens(word_ipa, merge_vowels=False)
                else:
                    phones = ['<unk>']
                
            
                
                try:
                    # crop local probabilities
                    local_probs = probs[start_frame:end_frame]
                    gt_mat_p, utt_idx_p = ctc_segmentation.prepare_text(self.config, phones)
                    timings_p, char_probs_p, _ = (
                        ctc_segmentation.ctc_segmentation(
                            self.config,
                            local_probs,
                            gt_mat_p
                        )
                    )
                
            
                    phone_segments = (
                        ctc_segmentation.determine_utterance_segments(
                            self.config,
                            utt_idx_p,
                            char_probs_p,
                            timings_p,
                            phones
                        )
                    )
                    
                except:
                    unit = max(0.001, (end_frame - start_frame)*self.config.index_duration/len(phones))
                    phone_segments = [(i*unit, (i+1)*unit, 0.0) for i, ph in enumerate(phones)]
            
                # ---------------------------------
                # convert local -> global timestamps
                # ---------------------------------
                word_phone_out = []
            
                for ph, p in zip(phones, phone_segments):
            
                    global_start = (p[0]+ start_frame * self.config.index_duration)
            
                    global_end = (p[1]+ start_frame * self.config.index_duration)
            
                    global_start = float(global_start)
                    global_end = float(global_end)

            
                    word_phone_out.append(
                        {
                            "text": ph,
                            "start": global_start,
                            "end": global_end,
                            "conf": round(float(p[2]), 3),
                        }
                    )
            
                phone_out.extend(word_phone_out)
                

            alignments.append({
                "words": word_out,
                "phones": phone_out,
            })
        return alignments

In [5]:
from praatio import textgrid


def _build_intervals_with_pauses(
    segments,
    xmin,
    xmax,
    min_gap=0.001,
):
    """
    Fill gaps between segments with silence intervals,
    while enforcing:
      - monotonic intervals
      - no overlaps
      - minimum duration/gap

    segments: list of dicts with keys:
        [start, end, text]
    """

    intervals = []

    # sort for safety
    segments = sorted(segments, key=lambda x: x["start"])

    cur = float(xmin)

    for seg in segments:

        start = round(float(seg["start"]), 3)
        end = round(float(seg["end"]), 3)
        text = seg["text"]

        # ---------------------------------
        # prevent backward movement
        # ---------------------------------
        if start < cur:
            start = cur

        # ---------------------------------
        # enforce minimum duration
        # ---------------------------------
        if end <= start:
            end = start + min_gap

        # ---------------------------------
        # insert silence gap if needed
        # ---------------------------------
        if start - cur >= min_gap:
            intervals.append((round(cur, 3), round(start, 3),""))

        # ---------------------------------
        # add segment
        # ---------------------------------
        intervals.append((round(start, 3),round(end, 3),text))

        # next interval must begin AFTER this
        cur = end

    # ---------------------------------
    # tail silence
    # ---------------------------------
    if xmax - cur >= min_gap:
        intervals.append((round(cur, 3),round(float(xmax), 3),""))

    return intervals


def save_textgrids(
    audio_filenames,
    alignments,
    out_dir,
    audio_durs,
):
    """
    Args:
        audio_paths: List[str]
        alignments: List[dict] with keys "words", "phones"
        out_dir: output directory
    """

    os.makedirs(out_dir, exist_ok=True)

    for filename, align, audio_dur in zip(audio_filenames, alignments, audio_durs):
        words = align["words"]
        phones = align["phones"]

        xmax = audio_dur

        xmin = 0.0

        # build tiers (with pauses filled)
        word_intervals = _build_intervals_with_pauses(words, xmin, xmax)
        phone_intervals = _build_intervals_with_pauses(phones, xmin, xmax)

        # create TextGrid
        tg = textgrid.Textgrid()

        word_tier = textgrid.IntervalTier(
            name="words",
            entries=word_intervals,
            minT=xmin,
            maxT=xmax,
        )

        phone_tier = textgrid.IntervalTier(
            name="phones",
            entries=phone_intervals,
            minT=xmin,
            maxT=xmax,
        )

        tg.addTier(word_tier)
        tg.addTier(phone_tier)

        # output filename
        name = os.path.splitext(filename)[0] + ".TextGrid"
        out_path = os.path.join(out_dir, name)

        # save
        tg.save(out_path, format="short_textgrid", includeBlankSpaces=True)

In [6]:
from itertools import islice
from tqdm.auto import tqdm

device = "cuda:0"

if model:
    model.eval()
    model.to(device)


def batch_iterator(iterable, batch_size):
    iterator = iter(iterable)

    while True:
        batch = list(islice(iterator, batch_size))

        if not batch:
            break

        yield batch


output_path = os.path.join("alignments", os.path.basename(model_path) if model_path[-1] != '/' else os.path.basename(model_path[:-1]))
os.makedirs(output_path, exist_ok=True)

In [7]:
import torch

if processor:
    ctc_segmentor = CTCSegmentation(tokenizer=processor.tokenizer)
else:
    ctc_segmentor = CTCSegmentation()

def detect_speech_bounds(audio, factor=0.05):
    energy = np.abs(audio)

    # smooth over ~25ms
    smooth = np.convolve(
        energy,
        np.ones(400) / 400,
        mode="same"
    )

    threshold = factor * np.percentile(smooth, 95)

    idx = np.where(smooth > threshold)[0]

    if len(idx) == 0:
        return 0, len(audio)

    return idx[0], idx[-1]
    
for lang in tqdm(LANGS):
    dataset = load_dataset("fleurs", lang, streaming=True, download_config=download_config, trust_remote_code=True)
    test_dataset = dataset["test"]   # streaming dataset
    if model:
        model.load_adapter(lang)
    alignments = []
    audio_filenames = []
    audio_durs = []
    for batch in batch_iterator(test_dataset, batch_size=2):

        # audio arrays
        audios = [x["audio"]["array"][:360000] for x in batch]

        if model:
            # processor handles padding dynamically
            inputs = processor(
                audios,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True,
            )
    
            inputs = {k: v.to(device) for k, v in inputs.items()}
            audio_lengths = inputs["attention_mask"].sum(-1)
            logit_lengths = model._get_feat_extract_output_lengths(audio_lengths).cpu().numpy()
            audio_lengths = audio_lengths.cpu().numpy()
            with torch.no_grad():
                logits = model(**inputs).logits
    
            probs_batch = torch.softmax(logits, dim=-1).cpu().numpy()
            
        else:
            audio_lengths = []
            probs_batch = []
            logit_lengths = []
            speech_offsets = []
        
            for audio in audios:
                start, end = detect_speech_bounds(audio)
                speech_offsets.append((start, end))
                audio_lengths.append(end - start)
                probs_batch.append(None)
                logit_lengths.append(100)

        alignments_batch = ctc_segmentor.get_word_and_timestamps_batch(
                probs_batch=probs_batch,
                audio_lens=audio_lengths,
                frame_lens=logit_lengths,
                ipa_transcripts=[x['ipa'] for x in batch],
                transcripts=[x['word_segmented'] for x in batch]
            )

        if not model:
            for alignment, (start_sample, _) in zip(alignments_batch, speech_offsets):
    
                offset_sec = start_sample / 16000
            
                for item in alignment["words"]:
                    item["start"] = item["start"] + offset_sec
                    item["end"] = item["end"] + offset_sec
            
                for item in alignment["phones"]:
                    item["start"] = item["start"] + offset_sec
                    item["end"] = item["end"] + offset_sec

        alignments.extend(alignments_batch)
        audio_durs.extend([ audio.shape[0] / 16000 for audio in audios])
        audio_filenames.extend(
            [os.path.basename(x["audio"]["path"]) for x in batch]
        )
    save_textgrids(
        audio_filenames,
        alignments,
        os.path.join(output_path, lang),
        audio_durs
    )  

100%|████████████████████████████████████████| 85/85 [2:25:21<00:00, 102.61s/it]


In [9]:
ctc_segmentor.total_failures

11

In [40]:
# Total failures
# w2v2-* - 11 
# mms-* - 11

## Qwen Alignments

In [8]:
import os
import textgrid
import torch

from glob import glob
from tqdm import tqdm
from qwen_asr import Qwen3ForcedAligner

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
SAMPLE_SIZE=250

QWEN3_FA_LANGS={
    "cmn_hans_cn":"Chinese",
    "de_de":"German",
    "en_us":"English",
    "es_419":"Spanish",
    "fr_fr":"French",
    "it_it":"Italian",
    "ja_jp":"Japanese",
    "ko_kr":"Korean",
    "pt_br":"Portuguese",
    "ru_ru":"Russian",
}

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------
model=Qwen3ForcedAligner.from_pretrained(
    "Qwen/Qwen3-ForcedAligner-0.6B",
    dtype=torch.bfloat16,
    device_map="cuda:0",
)

# ---------------------------------------------------------
# SAVE TG
# ---------------------------------------------------------
def save_word_textgrid(
    out_path,
    words,
    max_time
):

    os.makedirs(
        os.path.dirname(out_path),
        exist_ok=True
    )

    tg=textgrid.TextGrid(maxTime=max_time)

    tier=textgrid.IntervalTier(
        name="words",
        minTime=0.0,
        maxTime=max_time
    )

    for w in words:

        s=max(0.0,float(w["start"]))
        e=min(max_time,float(w["end"]))

        if e<=s:
            continue

        tier.add(
            s,
            e,
            w["text"]
        )

    tg.append(tier)
    tg.write(out_path)

# ---------------------------------------------------------
# RUN
# ---------------------------------------------------------
for lang,qwen_lang in QWEN3_FA_LANGS.items():

    print(f"\n--- {lang} ---")

    ds=load_dataset(
        "fleurs",
        lang,
        split="test",
        streaming=True,
        trust_remote_code=True,
        download_config=download_config,
    )


    out_root=os.path.join(
        "alignments/qwen3-FA",
        lang
    )

    success=0
    failed=0

    for x in tqdm(ds):

        try:

            audio=x["audio"]["array"][:360000]
            wav_path = x["audio"]["path"]
            text=x["word_segmented"]

            results=model.align(
                audio=(audio,16000),
                text=text,
                language=qwen_lang,
            )

            # batch size 1
            aligned=results[0]

            words=[]

            for tok in aligned:

                words.append({
                    "text":tok.text,
                    "start":tok.start_time,
                    "end":tok.end_time,
                })

            rel=os.path.splitext(
                os.path.basename(wav_path)
            )[0]+".TextGrid"

            out_path=os.path.join(
                out_root,
                rel
            )

            save_word_textgrid(
                out_path,
                words,
                max_time=len(audio)/16000.0
            )

            success+=1

        except Exception as e:

            failed+=1
            print(lang,e)

    print(
        f"{lang}: "
        f"{success} success "
        f"{failed} failed"
    )


--- cmn_hans_cn ---


945it [10:38,  1.48it/s]


cmn_hans_cn: 945 success 0 failed

--- de_de ---


862it [09:59,  1.44it/s]


de_de: 862 success 0 failed

--- en_us ---


...

ko_kr: 382 success 0 failed

--- pt_br ---


919it [10:11,  1.50it/s]


pt_br: 919 success 0 failed

--- ru_ru ---


775it [08:43,  1.48it/s]

ru_ru: 775 success 0 failed


## Doreco Alignments

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
from ctc_align import alignData

/localscratch/nnsfn01/anaconda3/envs/north_caucasus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
alignData(dataset='doreco_dataset', align_dir="alignments", model_path='models/w2v2-lv-60-espeak-ipa-doreco', trim_silence=True)

100%|███████████████████████████████████████████| 45/45 [13:53<00:00, 18.52s/it]

Total failures: 0


In [13]:
audio_path = "alignments/random_doreco/stan1290/audio/stan1290_00000014.wav"
tg_path = "alignments/mms-300m-ipa-doreco-SIL/stan1290/stan1290_00000014.TextGrid"

In [14]:
from IPython.display import Audio, display
import soundfile as sf
from textgrid import TextGrid

# -------- print TextGrid --------

tg = TextGrid.fromFile(tg_path)

for tier in tg.tiers:
    print(f"\n=== {tier.name} ===")
    for interval in tier:
        print(
            f"{interval.minTime:.3f}\t"
            f"{interval.maxTime:.3f}\t"
            f"{repr(interval.mark)}"
        )


=== words ===
0.000	0.039	''
0.039	0.371	'justement'
0.371	0.451	'à'
0.451	0.452	''
0.452	0.612	'chaque'
0.612	0.752	'fois'
0.752	0.753	''
0.753	0.873	'ils'
0.873	1.154	'essayaient'
1.154	1.334	'de'
1.334	1.465	''

=== phones ===
0.000	0.025	''
0.025	0.045	'Z'
0.045	0.090	'y'
0.090	0.135	's'
0.135	0.181	't'
0.181	0.226	'@'
0.226	0.271	'm'
0.271	0.316	'a'
0.316	0.361	'~'
0.361	0.371	''
0.371	0.411	'a'
0.411	0.452	''
0.452	0.492	'S'
0.492	0.532	'a'
0.532	0.572	'k'
0.572	0.602	''
0.602	0.622	'f'
0.622	0.649	''
0.649	0.696	'w'
0.696	0.743	'a'
0.743	0.753	''
0.753	0.793	'i'
0.793	0.833	'l'
0.833	0.873	''
0.873	0.913	'e'
0.913	0.953	's'
0.953	1.013	'E'
1.013	1.054	'j'
1.054	1.114	'E'
1.114	1.154	''
1.154	1.234	'd'
1.234	1.274	'@'
1.274	1.465	''


In [8]:
# -------- play audio segment --------
x = 0.904   # start time (s)
y = 1.244   # end time (s)

audio, sr = sf.read(audio_path)

segment = audio[int(x * sr): int(y * sr)]

display(Audio(segment, rate=sr))